# **Online Retail Business Analysis**

## Intermediate-Level Data Analytics Project

This project analyses an online retail transaction dataset to understand
customer behaviour, revenue generation, product performance, and sales trends.

The analysis will progress from data cleaning and exploratory analysis
towards customer-level and business-level insights, followed by
visual reporting and actionable recommendations.

### Key Business Objectives

- Understand the structure and quality of the retail transaction data.
- Analyse revenue and product performance.
- Examine customer purchasing behaviour and repeat activity.
- Identify important trends across time and countries.
- Identify patterns that may support business decision-making.
- Translate analytical findings into practical recommendations.

## 1. Project Setup

We begin by importing the Python libraries required for data manipulation,
numerical analysis, visualisation, and statistical exploration.

The initial setup is intentionally limited to libraries that directly support
the analytical workflow.

In [2]:
# Import the core libraries required for data analysis and visualization.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Load the raw Online Retail Excel dataset into a pandas DataFrame.

df = pd.read_excel("../data/raw/Online Retail.xlsx")

In [4]:
# Display the first five records to verify that the dataset loaded correctly.

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [5]:
# Display the number of rows and columns in the dataset.

df.shape

(541909, 8)

In [6]:
# Display the column names to understand the available analytical variables.

df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')

In [7]:
# Inspect data types, non-null counts, and memory usage for every column.

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [8]:
# Generate descriptive statistics for the numerical variables in the dataset.

df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


In [9]:
# Calculate the number and percentage of missing values in each column.

missing_values = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": df.isnull().mean() * 100
})

missing_values.sort_values("Missing Count", ascending=False)

,Missing Count,Missing Percentage
CustomerID,135080,24.926694
Description,1454,0.268311
InvoiceNo,0,0.000000
StockCode,0,0.000000
Quantity,0,0.000000
InvoiceDate,0,0.000000
UnitPrice,0,0.000000
Country,0,0.000000


In [10]:
# Count completely duplicated transaction records in the dataset.

df.duplicated().sum()

np.int64(5268)

## Section 1 — Dataset Assessment: Findings

The Online Retail dataset contains 541,909 transaction-line records across
8 variables, providing sufficient scale and detail for intermediate-level
business analysis.

The dataset contains substantial missingness in CustomerID, with 135,080
records (24.93%) lacking customer identification. This is particularly
important for customer-behaviour analysis because customer-level metrics
cannot be reliably calculated for unidentified transactions. Description
has comparatively minor missingness of 1,454 records (0.27%).

The numerical inspection also reveals unusual values. Quantity ranges from
-80,995 to 80,995, while UnitPrice includes negative and extremely large
values. These observations indicate that the dataset may contain returns,
cancellations, or other non-standard transactions that must be investigated
before revenue and customer metrics are calculated.

Additionally, 5,268 completely duplicated rows are present.

Therefore, the next stage will focus on understanding these data-quality
issues and determining appropriate business-aware cleaning rules rather
than removing records blindly.

## 2. Data Quality Investigation

Before cleaning the dataset, we investigate unusual observations to
understand their business meaning.

Negative quantities may represent cancelled or returned purchases rather
than erroneous records. Understanding these transactions is essential because
removing them without investigation could distort revenue, transaction, and
customer-level analysis.

We will first identify the frequency and characteristics of negative-
quantity transactions.

In [11]:
# Count transactions containing negative quantities to identify potential returns or cancellations.

negative_quantity = df[df["Quantity"] < 0]
negative_quantity.shape

(10624, 8)

In [12]:
# Display representative negative-quantity transactions for investigation.

negative_quantity.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-12-01 10:24:00,3.45,17548.0,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-12-01 12:38:00,4.25,17897.0,United Kingdom


In [13]:
# Summarize the distribution of quantities for transactions with negative quantities.

negative_quantity["Quantity"].describe()

count    10624.000000
mean       -45.607210
std       1092.214216
min     -80995.000000
25%        -10.000000
50%         -2.000000
75%         -1.000000
max         -1.000000
Name: Quantity, dtype: float64

In [14]:
# Identify invoice numbers associated with cancellation indicators.

df["InvoiceNo"].astype(str).str.startswith("C").value_counts()

InvoiceNo
False    532621
True       9288
Name: count, dtype: int64

In [15]:
# Display sample cancellation-coded invoices and their associated transaction details.

df[df["InvoiceNo"].astype(str).str.startswith("C")].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-12-01 10:24:00,3.45,17548.0,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-12-01 12:38:00,4.25,17897.0,United Kingdom


In [16]:
# Identify transactions containing negative unit prices for further investigation.

negative_price = df[df["UnitPrice"] < 0]
negative_price.shape

(2, 8)

In [17]:
# Display transactions with negative unit prices to understand their business context.

negative_price

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [18]:
# Count transactions with zero unit prices to identify potentially unusual or promotional records.

(df["UnitPrice"] == 0).sum()

np.int64(2515)

In [19]:
# Display examples of zero-priced transactions to determine their business context.

df[df["UnitPrice"] == 0].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom


In [20]:
# Extract completely duplicated records for detailed inspection.

duplicates = df[df.duplicated(keep=False)]
duplicates.head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
548,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom


In [21]:
# Count duplicate records grouped by invoice and product to identify recurring duplicate patterns.

duplicates.groupby(["InvoiceNo", "StockCode"]).size().sort_values(ascending=False).head(10)

InvoiceNo  StockCode
555524     22698        20
           22697        12
572861     22775         8
572344     M             6
541266     21754         6
538514     21756         6
541266     21755         6
578289     23395         6
540524     21756         6
547712     22699         5
dtype: int64

## Section 2 — Data Quality Investigation: Conclusions

The raw dataset contains several data-quality issues that must be addressed before conducting reliable business analysis. Approximately 24.93% of CustomerID values are missing, which limits direct customer-level analysis for those records, while missing product descriptions are comparatively minor at 0.27%.

The dataset also contains 10,624 transactions with negative quantities and 9,288 cancellation-coded invoice records, indicating substantial return or cancellation activity. These records should not be automatically discarded because they may provide valuable information about customer returns and transaction reversals.

Two transactions contain negative unit prices associated with "Adjust bad debt", suggesting accounting adjustments rather than genuine retail sales. In addition, 2,515 transactions have zero unit prices and therefore require contextual investigation before deciding whether they represent promotional, complimentary, or anomalous transactions.

Finally, 5,268 completely duplicated records were identified. These are strong candidates for removal during the cleaning stage, while repeated invoice-product combinations require more careful interpretation because repetition does not necessarily imply erroneous duplication.

Overall, the dataset is sufficiently rich, but a structured cleaning and transformation process is necessary before calculating revenue, customer behaviour, product performance, and other business metrics.

## 3. Data Cleaning & Preparation

Now that we have assessed the dataset and investigated its quality issues, we can start preparing it for actual business analysis.

The objective here is not simply to delete problematic rows. We will make deliberate cleaning decisions so that the final dataset is reliable for revenue, customer, product, regional, and trend analysis

### Preserve the original working data

In [22]:
# Create a separate working copy so the original raw dataset remains unchanged.
df_clean = df.copy()
df_clean.shape

(541909, 8)

In [23]:
# Store the original row count so that data reduction can be measured throughout cleaning.
original_rows = len(df_clean)
original_rows

541909

### Removing Exact Duplicate Records

In [24]:
# Count the exact duplicate records currently present in the working dataset.
df_clean.duplicated().sum()

np.int64(5268)

In [25]:
# Remove completely duplicated records while retaining the first occurrence of each record.
df_clean = df_clean.drop_duplicates()

In [26]:
# Display the dimensions of the cleaned dataset after removing exact duplicates.
df_clean.shape

(536641, 8)

In [27]:
# Verify that no completely duplicated records remain after the cleaning operation.
df_clean.duplicated().sum()

np.int64(0)

In [28]:
# Calculate how many records were removed during duplicate cleaning.
df.shape[0] - df_clean.shape[0]

5268

### Separate returns and cancellations before removing them

In [29]:
# Identify transactions with negative quantities so that returns can be preserved separately.
df_returns = df_clean[df_clean["Quantity"] < 0].copy()

In [30]:
# Record the number of return-related transaction rows preserved for separate analysis.
len(df_returns)

10587

In [31]:
# Calculate the total quantity represented by negative-quantity transactions.
df_returns["Quantity"].sum()

np.int64(-482517)

In [32]:
# Calculate the monetary value represented by negative-quantity transactions.
(df_returns["Quantity"] * df_returns["UnitPrice"]).sum()

np.float64(-893979.73)

### Remove non-sales transactions from the primary analytical dataset
A valid sale for the primary sales analysis is a transaction with positive quantity and positive unit price.

In [33]:
# Retain only positive-quantity and positive-price transactions for the primary sales analysis.
df_clean = df_clean[
    (df_clean["Quantity"] > 0) & (df_clean["UnitPrice"] > 0)
    ].copy()

In [34]:
# Display the dimensions of the dataset after removing non-sales transactions.
df_clean.shape

(524878, 8)

In [35]:
# Calculate how many rows were removed when restricting the dataset to valid sales transactions.
original_rows - len(df_clean)

17031

### Remove records with missing product descriptions

In [36]:
# Count missing product descriptions remaining after the sales transaction filtering.
df_clean["Description"].isna().sum()

np.int64(0)

In [37]:
# Remove transactions where the product description is unavailable.
df_clean = df_clean.dropna(subset=["Description"]).copy()

In [38]:
# Display the dimensions of the dataset after removing records with missing descriptions.
df_clean.shape

(524878, 8)

### Handle missing Customer IDs appropriately

In [39]:
# Create a customer-analysis dataset containing only transactions with identifiable customers.
df_customer = df_clean.dropna(subset=["CustomerID"]).copy()

In [40]:
# Display the dimensions of the customer-level dataset.
df_customer.shape

(392692, 8)

In [41]:
# Calculate the percentage of valid sales transactions that can be attributed to identifiable customers.
len(df_customer) / len(df_clean) * 100

74.81586197173439

### Create the revenue variable

In [42]:
# Calculate transaction-level revenue using quantity sold multiplied by unit price.
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["UnitPrice"]

In [43]:
# Verify the newly created revenue variable using a sample of transactions.
df_clean[["Quantity", "UnitPrice", "Revenue"]].head()

,Quantity,UnitPrice,Revenue
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


In [44]:
# Generate descriptive statistics for transaction-level revenue.
df_clean["Revenue"].describe()

count    524878.000000
mean         20.275399
std         271.693566
min           0.001000
25%           3.900000
50%           9.920000
75%          17.700000
max      168469.600000
Name: Revenue, dtype: float64

### Final validation of the cleaned dataset

In [45]:
# Display the final dimensions of the cleaned sales dataset.
df_clean.shape

(524878, 9)

In [46]:
# Count remaining missing values across all columns in the cleaned sales dataset.
df_clean.isna().sum()

InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     132186
Country             0
Revenue             0
dtype: int64

In [47]:
# Verify that no duplicate transaction records remain after cleaning.
df_clean.duplicated().sum()

np.int64(0)

In [48]:
# Verify that no negative quantities remain in the primary sales dataset.
(df_clean["Quantity"] < 0).sum()

np.int64(0)

In [49]:
# Verify that no zero or negative unit prices remain in the primary sales dataset.
(df_clean["UnitPrice"] <= 0).sum()

np.int64(0)

## Section 3 — Data Cleaning & Preparation: Conclusions

The raw dataset contained 541,909 transaction records. After removing 5,268 exact duplicates, preserving 10,587 negative-quantity transactions separately as return/cancellation records, and restricting the primary dataset to positive quantities and positive unit prices, we obtained 524,878 valid sales transactions.

Approximately 96.86% of the original records remain in the primary sales dataset, indicating that the majority of the data is suitable for sales analysis.

Missing product descriptions are no longer present after transaction filtering, while CustomerID remains missing for approximately 25.18% of valid sales transactions. Rather than discarding these otherwise valid sales, a separate df_customer dataset containing 392,692 customer-identifiable transactions was created for customer-level analysis.

A transaction-level Revenue variable was derived as Quantity × UnitPrice. Its mean (£20.28) substantially exceeds its median (£9.92), while the maximum (£168,469.60) is exceptionally large relative to the central distribution, indicating a strongly right-skewed revenue distribution and motivating later outlier investigation.

The final primary sales dataset contains no exact duplicates, negative quantities, or non-positive unit prices, making it suitable for subsequent exploratory and business analysis.

### Reset the index

In [50]:
# Reset the index of the cleaned dataset after row-level filtering.
df_clean = df_clean.reset_index(drop=True)
df_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [51]:
# Verify that the cleaned dataset now uses a continuous zero-based index.
df_clean.index

RangeIndex(start=0, stop=524878, step=1)

### Verify data types after cleaning

In [52]:
# Inspect the data types of all variables in the cleaned sales dataset.
df_clean.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
Revenue               float64
dtype: object

### Check the transaction date range

In [53]:
# Determine the earliest and latest transaction dates in the cleaned dataset.
df_clean["InvoiceDate"].agg(["min", "max"])

min   2010-12-01 08:26:00
max   2011-12-09 12:50:00
Name: InvoiceDate, dtype: datetime64[us]

### Inspect categorical consistency

In [54]:
# Count unique values in the principal categorical variables used for analysis.
df_clean[["StockCode", "Description", "Country"]].nunique()

StockCode      3922
Description    4026
Country          38
dtype: int64

In [55]:
# Display the distinct country names represented in the cleaned dataset.
df_clean["Country"].unique()

<StringArray>
[      'United Kingdom',               'France',            'Australia',
          'Netherlands',              'Germany',               'Norway',
                 'EIRE',          'Switzerland',                'Spain',
               'Poland',             'Portugal',                'Italy',
              'Belgium',            'Lithuania',                'Japan',
              'Iceland',      'Channel Islands',              'Denmark',
               'Cyprus',               'Sweden',              'Finland',
              'Austria',              'Bahrain',               'Israel',
               'Greece',            'Hong Kong',            'Singapore',
              'Lebanon', 'United Arab Emirates',         'Saudi Arabia',
       'Czech Republic',               'Canada',          'Unspecified',
               'Brazil',                  'USA',   'European Community',
                'Malta',                  'RSA']
Length: 38, dtype: str

### Check for whitespace inconsistencies in product descriptions

In [56]:
# Count product descriptions containing leading or trailing whitespace.
df_clean["Description"].str.contains(r"^\s|\s$", regex=True).sum()

np.int64(110582)

### Check whether invoice and stock codes contain suspicious blank values

In [57]:
# Count blank or whitespace-only invoice numbers in the cleaned dataset.
df_clean["InvoiceNo"].astype(str).str.strip().eq("").sum()

np.int64(0)

In [58]:
# Count blank or whitespace-only stock codes in the cleaned dataset.
df_clean["StockCode"].astype(str).str.strip().eq("").sum()

np.int64(0)

### Verify the revenue calculation once more

In [59]:
# Verify that the derived revenue variable contains no missing values.
df_clean["Revenue"].isna().sum()

np.int64(0)

In [60]:
# Verify that all calculated revenue values are positive for valid sales transactions.
(df_clean["Revenue"] <= 0).sum()

np.int64(0)

## Section 3 — Data Cleaning & Preparation: Deductions and Conclusions

Based on the outputs you actually provided, we can now conclude:

- The cleaned sales dataset contains 524,878 valid transaction records.
- The index has been reset successfully to a continuous zero-based index.
- The variables have appropriate data types for subsequent analysis.
- The transaction period runs from December 1, 2010 to December 9, 2011; therefore, December 2011 is only partially represented.
- The dataset contains 3,922 unique stock codes, 4,026 descriptions, and 38 countries.
- There are 110,582 records with leading/trailing whitespace in product descriptions, which is a genuine formatting inconsistency that should be cleaned.
- There are no blank invoice numbers or stock codes.
- Revenue is completely populated and every retained transaction has positive revenue.
- Return/cancellation transactions were not deleted blindly; they were preserved separately in df_returns before creating the primary sales dataset. That's an important analytical decision because returns contain business information of their own.
- CustomerID is still missing for a substantial portion of transactions, but we correctly preserved a separate df_customer dataset for analyses requiring identifiable customers rather than unnecessarily deleting those transactions from the primary sales dataset.

### Final Cleaning & Sanity Check

In [61]:
# Remove leading and trailing whitespace from product descriptions.
df_clean["Description"] = df_clean["Description"].str.strip()

In [62]:
# Verify that no product descriptions contain leading or trailing whitespace.
df_clean["Description"].str.contains(r"^\s|\s$", regex=True).sum()

np.int64(0)

### Synchronize the customer-level dataset

In [63]:
# Recreate the customer-analysis dataset from the cleaned sales dataset.
df_customer = df_clean.dropna(subset=["CustomerID"]).copy()
df_customer.shape

(392692, 9)

In [64]:
# Check remaining missing values in the primary sales dataset.
df_clean.isna().sum()

InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     132186
Country             0
Revenue             0
dtype: int64

### Final structural validation

In [65]:
# Verify the final structure and key data-quality conditions.
print("Shape:", df_clean.shape)
print("Duplicate rows:", df_clean.duplicated().sum())
print("Negative quantities:", (df_clean["Quantity"] < 0).sum())
print("Non-positive unit prices:", (df_clean["UnitPrice"] <= 0).sum())
print("Missing descriptions:", df_clean["Description"].isna().sum())
print("Missing revenue:", df_clean["Revenue"].isna().sum())

Shape: (524878, 9)
Duplicate rows: 0
Negative quantities: 0
Non-positive unit prices: 0
Missing descriptions: 0
Missing revenue: 0


## Section 3 — Data Cleaning & Preparation: Deductions and Conclusions

The raw dataset initially contained **541,909 transaction records across 8 variables**. After identifying and removing **5,268 exact duplicate records**, the working dataset was reduced to 536,641 records. Transactions with negative quantities were treated as return/cancellation-related records rather than being discarded blindly; **10,587 such transactions were preserved separately in `df_returns`**, allowing them to remain available for dedicated return analysis.

For the primary sales analysis, the dataset was restricted to transactions with **positive quantities and positive unit prices**, resulting in **524,878 valid sales transactions across 8 original variables**. This represents approximately **96.86% of the original dataset**, indicating that the vast majority of records are suitable for conventional sales analysis. The product descriptions were also standardized by removing leading and trailing whitespace, eliminating a formatting inconsistency previously identified in **110,582 records**. No blank or whitespace-only invoice numbers or stock codes were found.

A transaction-level **`Revenue`** variable was derived using `Quantity × UnitPrice`, expanding the primary dataset to **9 variables**. The resulting revenue distribution has a mean of approximately **£20.28** and a median of **£9.92**, while the maximum reaches **£168,469.60**. The substantial difference between the mean and median, together with the exceptionally large maximum, indicates a **strongly right-skewed revenue distribution** and suggests that high-value transactions should be investigated during subsequent exploratory analysis rather than assumed to be erroneous.

Missing-value analysis confirms that the final primary sales dataset contains **no missing values in InvoiceNo, StockCode, Description, Quantity, InvoiceDate, UnitPrice, Country, or Revenue**. However, **132,186 CustomerID values remain missing**, meaning that not every valid transaction can be attributed to an identifiable customer. These transactions were retained in the primary sales dataset because they remain valid sales records. A separate `df_customer` dataset was therefore created containing **392,692 customer-identifiable transactions**, representing approximately **74.82% of the cleaned sales transactions**, and will be used for customer-level analysis.

The cleaned dataset covers transactions from **December 1, 2010 to December 9, 2011**, meaning that the final month of the observed period is only partially represented and should be considered when interpreting time-based trends. The dataset contains **3,922 unique StockCodes, 4,026 unique product descriptions, and 38 countries**, providing sufficient categorical diversity for product, geographic, and customer-oriented analysis.

Finally, the cleaned primary sales dataset passed the major structural validation checks: **0 exact duplicates, 0 negative quantities, 0 non-positive unit prices, 0 missing descriptions, and 0 missing revenue values**. The dataset is therefore sufficiently clean and structurally consistent for the next stages of analysis, while returns, unidentified customers, and extreme revenue values have been preserved or isolated in a manner that prevents loss of potentially valuable business information.